# PAN dataset analysis

Facts of the raw PAN 2020 training pairs and the PAN 2021 test pairs, quoted in the thesis.
Every number is computed from the raw files; no text is printed.

In [ ]:
RAW_DIR_NAME = "raw"

## Imports

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.figure import Figure

from avcore import data, tracking

## Configuration

In [ ]:
PROJECT_DIR = Path.cwd().parents[1]
RAW_DIR = PROJECT_DIR / "data" / RAW_DIR_NAME
OUTPUT_DIR = PROJECT_DIR / "outputs" / "dataset_analysis"

SPLIT_FILES = {
    "train": (data.TRAIN_PAIRS_FILE, data.TRAIN_TRUTH_FILE),
    "test": (data.TEST_PAIRS_FILE, data.TEST_TRUTH_FILE),
}
# the histogram range and bins of the v1 figures
HISTOGRAM_RANGE = (15_000, 23_000)
HISTOGRAM_BINS = 100
HISTOGRAM_TITLES = {
    "train": "Distribuce délek textů pro trénovací dataset",
    "test": "Distribuce délek textů pro testovací dataset",
}

## Run context

The commit, the clean repository check and the sha256 of the four raw files.

In [ ]:
raw_paths = [RAW_DIR / name for files in SPLIT_FILES.values() for name in files]
context = tracking.collect_run_context(PROJECT_DIR, raw_paths)
print(f"commit {context.environment['commit']}")
for path in raw_paths:
    print(f"{path.name} {context.environment[f'data_sha256.{path.name}'][:16]}")

## Read the pairs

One row per line of the pairs file; line i of the truth file holds the label of line i.
The texts are kept only as a hash and a length, so the 12 GB training file never sits in memory.

In [ ]:
def hash_text(text: str) -> str:
    """Return the sha256 of a text.

    Args:
        text: The text.

    Returns:
        The hex digest of the UTF-8 bytes.
    """
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_pair_records(pairs_path: Path, truth_path: Path) -> pd.DataFrame:
    """Read a PAN pairs file and its truth file into one row per pair.

    Args:
        pairs_path: The pairs jsonl file (id, fandoms, pair).
        truth_path: The truth jsonl file (id, same, authors).

    Returns:
        The columns id, same, author_1, author_2, fandom_1, fandom_2, text_hash_1,
        text_hash_2, length_1 and length_2, in the order of the pairs file.

    Raises:
        ValueError: If the ids on a line differ or a pair does not have exactly 2 texts.
    """
    records = []
    with (
        pairs_path.open(encoding="utf-8") as pairs_file,
        truth_path.open(encoding="utf-8") as truth_file,
    ):
        # strict=True raises if one file has more lines than the other
        for line_number, (pair_line, truth_line) in enumerate(
            zip(pairs_file, truth_file, strict=True), start=1
        ):
            pair = json.loads(pair_line)
            truth = json.loads(truth_line)
            if pair["id"] != truth["id"]:
                raise ValueError(
                    f"line {line_number}: id {pair['id']} does not match truth id {truth['id']}"
                )
            if len(pair["pair"]) != 2:
                raise ValueError(f"line {line_number}: pair is not a list of 2 texts")
            first_text, second_text = pair["pair"]
            records.append(
                {
                    "id": pair["id"],
                    "same": truth["same"],
                    "author_1": truth["authors"][0],
                    "author_2": truth["authors"][1],
                    "fandom_1": pair["fandoms"][0],
                    "fandom_2": pair["fandoms"][1],
                    "text_hash_1": hash_text(first_text),
                    "text_hash_2": hash_text(second_text),
                    "length_1": len(first_text),
                    "length_2": len(second_text),
                }
            )
    return pd.DataFrame(records)

In [ ]:
records_dfs = {}
for split, (pairs_file, truth_file) in SPLIT_FILES.items():
    records_df = read_pair_records(RAW_DIR / pairs_file, RAW_DIR / truth_file)
    records_dfs[split] = records_df
    print(f"{split}: {len(records_df)} pairs")

## Statistics of each split

In [ ]:
def count_repeats(values: pd.Series) -> dict[str, int]:
    """Count the values that occur more than once.

    Args:
        values: Hashable values, one per row.

    Returns:
        groups - values that occur more than once; extra_rows - rows beyond the first of each value.
    """
    counts = values.value_counts()
    repeated = counts[counts > 1]
    return {"groups": int(len(repeated)), "extra_rows": int((repeated - 1).sum())}


def compute_split_statistics(records_df: pd.DataFrame) -> dict[str, int | float]:
    """Compute the dataset facts of one split.

    Args:
        records_df: The rows of read_pair_records.

    Returns:
        The facts by name; lengths are in characters of the raw texts.
    """
    lengths = pd.concat([records_df["length_1"], records_df["length_2"]])
    text_hashes = pd.concat([records_df["text_hash_1"], records_df["text_hash_2"]])
    ordered_pairs = records_df["text_hash_1"] + records_df["text_hash_2"]
    # a pair and its swapped version count as the same pair
    unordered_pairs = records_df[["text_hash_1", "text_hash_2"]].apply(
        lambda row: "".join(sorted(row)), axis=1
    )
    authors = pd.concat([records_df["author_1"], records_df["author_2"]])
    same_fandom = records_df["fandom_1"] == records_df["fandom_2"]
    ids = count_repeats(records_df["id"])
    texts = count_repeats(text_hashes)
    ordered = count_repeats(ordered_pairs)
    unordered = count_repeats(unordered_pairs)
    return {
        "pairs": len(records_df),
        "same_author_pairs": int(records_df["same"].sum()),
        "different_author_pairs": int((~records_df["same"]).sum()),
        "unique_ids": int(records_df["id"].nunique()),
        "repeated_ids": ids["groups"],
        "repeated_id_extra_rows": ids["extra_rows"],
        "texts": len(text_hashes),
        "unique_texts": int(text_hashes.nunique()),
        "repeated_texts": texts["groups"],
        "duplicate_pairs_ordered": ordered["groups"],
        "duplicate_pairs_ordered_extra_rows": ordered["extra_rows"],
        "duplicate_pairs_unordered": unordered["groups"],
        "duplicate_pairs_unordered_extra_rows": unordered["extra_rows"],
        "length_min": int(lengths.min()),
        "length_max": int(lengths.max()),
        "length_median": float(lengths.median()),
        "length_mean": float(lengths.mean()),
        "authors": int(authors.nunique()),
        "same_author_pair_authors": int(
            records_df.loc[records_df["same"], "author_1"].nunique()
        ),
        "fandoms": int(
            pd.concat([records_df["fandom_1"], records_df["fandom_2"]]).nunique()
        ),
        "same_author_same_fandom": int((records_df["same"] & same_fandom).sum()),
        "same_author_cross_fandom": int((records_df["same"] & ~same_fandom).sum()),
        "different_author_same_fandom": int((~records_df["same"] & same_fandom).sum()),
        "different_author_cross_fandom": int(
            (~records_df["same"] & ~same_fandom).sum()
        ),
    }

In [ ]:
split_statistics = {}
statistics_columns = {}
for split, records_df in records_dfs.items():
    split_statistics[split] = compute_split_statistics(records_df)
    # dtype=object keeps the counts as integers next to the float lengths
    statistics_columns[split] = pd.Series(split_statistics[split], dtype=object)
pd.DataFrame(statistics_columns)

## Overlap of the training and the test pairs

In [ ]:
def compute_overlap_statistics(
    train_records_df: pd.DataFrame, test_records_df: pd.DataFrame
) -> dict[str, int | float]:
    """Count what the training and the test pairs share.

    Args:
        train_records_df: The rows of read_pair_records for the training pairs.
        test_records_df: The rows of read_pair_records for the test pairs.

    Returns:
        The shared ids, authors, fandoms and texts, the total of pairs and the test
        share in percent cut off (not rounded) to 2 decimals.
    """

    def values(records_df: pd.DataFrame, prefix: str) -> set[str]:
        return set(records_df[f"{prefix}_1"]) | set(records_df[f"{prefix}_2"])

    total_pairs = len(train_records_df) + len(test_records_df)
    return {
        "shared_ids": len(set(train_records_df["id"]) & set(test_records_df["id"])),
        "shared_authors": len(
            values(train_records_df, "author") & values(test_records_df, "author")
        ),
        "shared_fandoms": len(
            values(train_records_df, "fandom") & values(test_records_df, "fandom")
        ),
        "shared_texts": len(
            values(train_records_df, "text_hash") & values(test_records_df, "text_hash")
        ),
        "total_pairs": total_pairs,
        "test_share_percent": int(len(test_records_df) * 10_000 / total_pairs) / 100,
    }

In [ ]:
overlap_statistics = compute_overlap_statistics(
    records_dfs["train"], records_dfs["test"]
)
# dtype=object keeps the counts as integers next to the float share
pd.Series(overlap_statistics, dtype=object)

## Text length histograms

In [ ]:
def plot_text_length_histogram(
    lengths: pd.Series,
    title: str,
    length_range: tuple[int, int],
    bins: int,
) -> Figure:
    """Plot the histogram of the text lengths inside a range.

    Args:
        lengths: Text lengths in characters.
        title: Title of the figure.
        length_range: Lengths strictly between these two values are plotted.
        bins: Number of bins.

    Returns:
        The figure.
    """
    low, high = length_range
    shown_lengths = lengths[(lengths > low) & (lengths < high)]
    figure, axes = plt.subplots()
    axes.hist(shown_lengths, bins=bins, edgecolor="black")
    axes.set_title(title)
    axes.set_xlabel("Délka textu (počet znaků)")
    axes.set_ylabel("Počet textů (frekvence)")
    figure.tight_layout()
    return figure

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
low, high = HISTOGRAM_RANGE
for split, records_df in records_dfs.items():
    lengths = pd.concat([records_df["length_1"], records_df["length_2"]])
    outside = int(((lengths <= low) | (lengths >= high)).sum())
    print(f"{split}: {outside} of {len(lengths)} texts outside {HISTOGRAM_RANGE}")
    figure = plot_text_length_histogram(
        lengths=lengths,
        title=HISTOGRAM_TITLES[split],
        length_range=HISTOGRAM_RANGE,
        bins=HISTOGRAM_BINS,
    )
    figure.savefig(OUTPUT_DIR / f"{split}_text_length_histogram.png")
    plt.show()

## Save the facts

In [ ]:
facts = {
    "commit": context.environment["commit"],
    "statistics": split_statistics,
    "overlap": overlap_statistics,
}
facts_path = OUTPUT_DIR / "dataset_facts.json"
facts_path.write_text(json.dumps(facts, indent=2), encoding="utf-8")
print(f"saved {facts_path}")